# NAS Results Analysis
Run while training is in progress or after completion.
Reads metrics.json files + evaluates the best saved model.

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 1 — Setup
# ══════════════════════════════════════════════════════════════════════════
import os, sys, glob, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings("ignore")

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))
sys.path[:0] = ["training", ".", "pyqnas/src"]

from pynas.core.config import load_default_config
from pynas.core.population import Population
from dataset import TileDataset
from datamodule import BalancedTileDataModule

SAVE_DIR     = "models_traced"
TILES_ROOT   = "processed/dataset_v1"
INDEX_CSV    = f"{TILES_ROOT}/tiles_index.csv"
NAMES        = ["clear","fresh_burn","old_burn","cloud","shadow","water"]

# DEVICE: "cuda:0" while the live search runs on GPU 1 only.
# Switch to "cpu" if generation 2+ starts dispatching to both GPUs,
# or if you see VRAM drop unexpectedly — see the guard in Cell 4.
DEVICE = "cuda:0"

config = load_default_config()

class LightDM: input_shape=(7,256,256); num_classes=6

# Bounds must match what actually trained this population — check config
# if these values drift from what Cell 5 of the training notebook verified.
pop = Population(n_individuals=30, max_layers=7, dm=LightDM(),
                 max_parameters=5_000_000, min_parameters=200_000,
                 save_directory=SAVE_DIR)
pop.cfg = config

tiles  = pd.read_csv(INDEX_CSV)
val_df = tiles[(tiles.split=="val") & (tiles.gsd=="native")].reset_index(drop=True)
ds     = TileDataset(TILES_ROOT, val_df, is_train=False)
print(f"Val set: {len(ds):,} tiles")

_pop_cache = {}
def load_population_for_gen(gen):
    """Mirrors nas_worker.py's fallback: per-gen pkl, else gen 0."""
    if gen in _pop_cache: return _pop_cache[gen]
    pkl = f"{SAVE_DIR}/src/population_{gen}.pkl"
    if not os.path.exists(pkl):
        pkl = f"{SAVE_DIR}/src/population_0.pkl"
    plist = pickle.load(open(pkl, "rb"))
    _pop_cache[gen] = plist
    return plist

print("Setup complete ✓")

Val set: 6,367 tiles
Setup complete ✓


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 2 (revised) — Architecture lookup with shape-based disambiguation
# ══════════════════════════════════════════════════════════════════════════
# model_size alone becomes ambiguous once evolve() starts breeding — crossover
# children drawn from a shrinking, fitness-sorted mating pool often collide on
# exact parameter count (expected GA convergence, not a bug). When that
# happens, disambiguate using the actual saved checkpoint's per-layer weight
# SHAPES: build each same-param candidate, compare its state_dict shape
# signature against the checkpoint's. If several candidates happen to be
# architecturally identical to each other, any one is safe to use (they build
# the same model). Only genuinely different architectures that both match the
# checkpoint's shape would be truly unresolvable — flagged loudly, not guessed.

def _shape_signature(state_dict):
    return tuple(sorted((k, tuple(v.shape)) for k, v in state_dict.items()))

def find_individual_by_params(gen, target_params, checkpoint_path=None, is_fp16aware=False):
    plist = load_population_for_gen(gen)
    matches = [ind for ind in plist if getattr(ind, "model_size", None) == target_params]
    if len(matches) == 0:
        raise ValueError(f"gen{gen}: no individual with model_size=={target_params:,}")
    if len(matches) == 1:
        return matches[0]

    if checkpoint_path is None or not os.path.exists(checkpoint_path):
        raise ValueError(f"gen{gen}: {len(matches)} individuals share model_size=={target_params:,} "
                          f"and no checkpoint provided to disambiguate")
    ckpt_state = torch.load(checkpoint_path, map_location="cpu")
    ckpt_sig = _shape_signature(ckpt_state)

    resolved = []
    seen_signatures = set()
    for ind in matches:
        try:
            model, _ = pop.build_model(ind.parsed_layers, task="segmentation")
            if is_fp16aware:
                from pynas.core.qat_utils import read_fp16aware_opts, prepare_fp16_aware
                prepare_fp16_aware(model, read_fp16aware_opts(config))
            sig = _shape_signature(model.state_dict())
            if sig == ckpt_sig:
                resolved.append(ind)
                seen_signatures.add(sig)
        except Exception:
            continue

    if len(resolved) == 0:
        raise ValueError(f"gen{gen}: {len(matches)} same-param individuals, none match checkpoint shape")
    if len(seen_signatures) > 1:
        raise ValueError(f"gen{gen}: multiple DIFFERENT architectures both match checkpoint shape — truly ambiguous")
    return resolved[0]  # all resolved candidates share one shape signature — interchangeable

print("find_individual_by_params (shape-disambiguating) defined ✓")

find_individual_by_params (shape-disambiguating) defined ✓


In [3]:
def load_fp32_model(gen, idx, device=DEVICE):
    mpath = f"{SAVE_DIR}/generation_{gen}/model_{idx}/metrics.json"
    if not os.path.exists(mpath): return None
    m = json.load(open(mpath))
    p = f"{SAVE_DIR}/generation_{gen}/model_{idx}/model_fp32.pt"
    if not os.path.exists(p): return None
    individual = find_individual_by_params(gen, m["params"], checkpoint_path=p, is_fp16aware=False)
    model, _ = pop.build_model(individual.parsed_layers, task="segmentation")
    state = torch.load(p, map_location=device)
    model.load_state_dict(state, strict=False)
    return model.to(device)

def load_fp16aware_model(gen, idx, device=DEVICE):
    from pynas.core.qat_utils import read_fp16aware_opts, prepare_fp16_aware
    mpath = f"{SAVE_DIR}/generation_{gen}/model_{idx}/metrics.json"
    if not os.path.exists(mpath): return None
    m = json.load(open(mpath))
    p = f"{SAVE_DIR}/generation_{gen}/model_{idx}/model_fp16aware.pt"
    if not os.path.exists(p): return None
    individual = find_individual_by_params(gen, m["params"], checkpoint_path=p, is_fp16aware=True)
    model, _ = pop.build_model(individual.parsed_layers, task="segmentation")
    model = model.to(device)
    ctx = prepare_fp16_aware(model, read_fp16aware_opts(config))
    state = torch.load(p, map_location=device)
    model.load_state_dict(state, strict=False)
    return model

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 4 — Per-class eval stats
# ══════════════════════════════════════════════════════════════════════════
def eval_model_stats(model, n=150, device=DEVICE):
    model = model.eval()
    I=np.zeros(6); U=np.zeros(6); P=np.zeros(6); G=np.zeros(6)
    with torch.no_grad():
        for i in range(min(n, len(ds))):
            img, mask = ds[i]
            pr = model(img.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
            gt = mask.numpy(); v = gt != -1
            for c in range(6):
                a = (pr==c)&v; b = (gt==c)&v
                I[c] += (a&b).sum(); U[c] += (a|b).sum()
                P[c] += a.sum();     G[c] += b.sum()
    iou = I / np.maximum(U, 1)
    total = max(G.sum(), 1)
    return G/total*100, P/total*100, iou

def collapse_flag(pred_pct):
    return "⚠ COLLAPSE" if pred_pct.max() > 85.0 else ""

print("eval_model_stats defined ✓")

eval_model_stats defined ✓


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 5 — Full panel: FP32 stage, every completed candidate
# ══════════════════════════════════════════════════════════════════════════
vram_guard()

rows = []
metric_files = sorted(glob.glob(f"{SAVE_DIR}/generation_*/model_*/metrics.json"))
print(f"Found {len(metric_files)} completed candidates\n")

for mf in metric_files:
    m = json.load(open(mf))
    gen, idx, params = m["gen"], m["idx"], m["params"]
    try:
        model = load_fp32_model(gen, idx)
        if model is None:
            print(f"[gen{gen} idx{idx}] fp32 model missing — skip"); continue
        gt_pct, pred_pct, iou = eval_model_stats(model, n=150)
        flag = collapse_flag(pred_pct)
        print(f"[gen{gen} idx{idx:>2}] params={params:>9,}  mean_IoU={iou.mean():.4f}  "
              f"fp32_iou(logged)={m['fp32_iou']:.4f}  {flag}")
        rows.append({"gen":gen,"idx":idx,"stage":"fp32","params":params,
                     "mean_iou":iou.mean(), "logged_iou":m["fp32_iou"],
                     "max_pred_pct":pred_pct.max(), "max_pred_class":NAMES[pred_pct.argmax()],
                     **{f"pred_{n}":p for n,p in zip(NAMES,pred_pct)},
                     **{f"iou_{n}":v for n,v in zip(NAMES,iou)}})
        del model
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    except Exception as e:
        print(f"[gen{gen} idx{idx}] FP32 eval FAILED: {e}")

fp32_df = pd.DataFrame(rows)
print(f"\n{len(fp32_df)} FP32 models evaluated")

NameError: name 'vram_guard' is not defined

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 6 — Full panel: FP16-aware stage, every completed candidate
# ══════════════════════════════════════════════════════════════════════════
vram_guard()

rows16 = []
for mf in metric_files:
    m = json.load(open(mf))
    gen, idx, params = m["gen"], m["idx"], m["params"]
    try:
        model = load_fp16aware_model(gen, idx)
        if model is None:
            print(f"[gen{gen} idx{idx}] fp16aware model missing — skip"); continue
        gt_pct, pred_pct, iou = eval_model_stats(model, n=150)
        flag = collapse_flag(pred_pct)
        print(f"[gen{gen} idx{idx:>2}] params={params:>9,}  mean_IoU={iou.mean():.4f}  "
              f"fp16_iou(logged)={m['fp16_iou']:.4f}  {flag}")
        rows16.append({"gen":gen,"idx":idx,"stage":"fp16aware","params":params,
                       "mean_iou":iou.mean(), "logged_iou":m["fp16_iou"],
                       "max_pred_pct":pred_pct.max(), "max_pred_class":NAMES[pred_pct.argmax()],
                       **{f"pred_{n}":p for n,p in zip(NAMES,pred_pct)},
                       **{f"iou_{n}":v for n,v in zip(NAMES,iou)}})
        del model
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    except Exception as e:
        print(f"[gen{gen} idx{idx}] FP16-aware eval FAILED: {e}")

fp16_df = pd.DataFrame(rows16)
print(f"\n{len(fp16_df)} FP16-aware models evaluated")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 7 — Combined summary + collapse audit
# ══════════════════════════════════════════════════════════════════════════
combined = pd.concat([fp32_df, fp16_df], ignore_index=True) if len(fp16_df) else fp32_df

summary = combined[["gen","idx","stage","params","mean_iou","logged_iou",
                     "max_pred_class","max_pred_pct"]].sort_values(["gen","idx","stage"])
print(summary.to_string(index=False))

n_collapsed = (combined["max_pred_pct"] > 85).sum()
print(f"\n{'='*60}\n  {n_collapsed}/{len(combined)} evaluations show one class >85% of predictions\n{'='*60}")
if n_collapsed:
    print(combined[combined["max_pred_pct"]>85][
        ["gen","idx","stage","params","max_pred_class","max_pred_pct","mean_iou"]
    ].to_string(index=False))

combined.to_csv(f"{SAVE_DIR}/per_class_diagnostic_full.csv", index=False)
print(f"\nSaved → {SAVE_DIR}/per_class_diagnostic_full.csv")

best = combined.sort_values("mean_iou", ascending=False).iloc[0]
print(f"\nBest by mean_iou: gen={best.gen} idx={best.idx} stage={best.stage} "
      f"params={best.params:,} mean_iou={best.mean_iou:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 8 — Rare-class confusion: where do shadow/cloud pixels actually end up?
# ══════════════════════════════════════════════════════════════════════════
# 0.0 IoU for cloud/shadow could mean "harmless, just unlearnable from so few
# examples" or "actively bleeding into fresh_burn as false positives." This
# distinguishes the two. Uses only tiles KNOWN to contain the class, since
# they're too rare (~1% of tiles) for random sampling to reliably hit.

val_df_full = tiles[(tiles.split=="val") & (tiles.gsd=="native")].reset_index(drop=True)
ds_check = TileDataset(TILES_ROOT, val_df_full, is_train=False)

shadow_tile_indices = val_df_full[val_df_full.get("frac_5", 0) > 0].index.tolist()
cloud_tile_indices  = val_df_full[val_df_full.get("frac_4", 0) > 0].index.tolist()
print(f"Shadow tiles: {len(shadow_tile_indices)}   Cloud tiles: {len(cloud_tile_indices)}")

def class_confusion(gen, idx, class_id, tile_indices, stage="fp32", device=DEVICE):
    model = load_fp32_model(gen, idx, device) if stage=="fp32" else load_fp16aware_model(gen, idx, device)
    if model is None: print("Model not found"); return
    model.eval()
    confusion = np.zeros(6); n_px = 0
    with torch.no_grad():
        for i in tile_indices:
            img, mask = ds_check[i]
            gt = mask.numpy()
            cls_mask = gt == class_id
            if cls_mask.sum() == 0: continue
            pr = model(img.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
            n_px += cls_mask.sum()
            for c in range(6):
                confusion[c] += ((pr == c) & cls_mask).sum()
    print(f"n_pixels={n_px} across {len(tile_indices)} tiles")
    for i, nm in enumerate(NAMES):
        print(f"  → {nm:<12}{confusion[i]/max(n_px,1)*100:>6.2f}%")
    return confusion / max(n_px, 1)

vram_guard()
best_gen, best_idx = int(best.gen), int(best.idx)

print(f"\n=== Shadow confusion (gen{best_gen} idx{best_idx}, fp32) ===")
class_confusion(best_gen, best_idx, class_id=4, tile_indices=shadow_tile_indices, stage="fp32")

print(f"\n=== Cloud confusion (gen{best_gen} idx{best_idx}, fp32) ===")
class_confusion(best_gen, best_idx, class_id=3, tile_indices=cloud_tile_indices, stage="fp32")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 9 — Sampler exposure check: is a class reaching training batches?
# ══════════════════════════════════════════════════════════════════════════
NAS_SUBSET_N = config.getint("GA", "nas_subset_n", fallback=15_000)

dm_check = BalancedTileDataModule(TILES_ROOT, INDEX_CSV, batch_size=16,
                                   num_workers=0, nas_subset_n=NAS_SUBSET_N)
dm_check.setup()
dl = dm_check.train_dataloader()
class_pixels = np.zeros(6)
N_BATCHES = 30
for i, (img, mask) in enumerate(dl):
    if i >= N_BATCHES: break
    for c in range(6):
        class_pixels[c] += (mask == c).sum().item()
total = class_pixels.sum()
print(f"Class share across {N_BATCHES} training batches:")
for i, nm in enumerate(NAMES):
    print(f"  {nm:<14}{class_pixels[i]/total*100:>6.2f}%")